# Data Loading V2

# Data Loading V2

## Load All Datasets

In [2]:
import pandas as pd
import numpy as np
import os

BASE = r'C:\Project\FIFA_World_Cup_2026'

results = pd.read_csv(os.path.join(BASE, 'Data', 'results.csv'))
goalscorers = pd.read_csv(os.path.join(BASE, 'Data', 'goalscorers.csv'))
elo = pd.read_csv(os.path.join(BASE, 'Data', 'elo_ratings_wc2026.csv'))
players = pd.read_csv(os.path.join(BASE, 'Data', 'players_data-2025_2026.csv'))
player_elo = pd.read_csv(os.path.join(BASE, 'Data', 'players.csv'))

print("Results shape:", results.shape)
print("Goalscorers shape:", goalscorers.shape)
print("ELO shape:", elo.shape)
print("Players shape:", players.shape)
print("Player ELO shape:", player_elo.shape)

Results shape: (49437, 9)
Goalscorers shape: (47601, 8)
ELO shape: (4683, 23)
Players shape: (2839, 102)
Player ELO shape: (70378, 16)


In [3]:
print("ELO columns:", elo.columns.tolist())
print("\nPlayers columns:", players.columns.tolist())
print("\nPlayer ELO columns:", player_elo.columns.tolist())
print("\nELO sample:")
print(elo.head(3))
print("\nPlayers sample:")
print(players.head(3))

ELO columns: ['year', 'snapshot_date', 'country', 'rank', 'country_code', 'rating', 'rank_max', 'rating_max', 'rank_avg', 'rating_avg', 'rank_min', 'rating_min', 'matches_total', 'matches_home', 'matches_away', 'matches_neutral', 'wins', 'losses', 'draws', 'goals_for', 'goals_against', 'confederation', 'is_host']

Players columns: ['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'G+A-PK', 'Rk_stats_keeper', 'Nation_stats_keeper', 'Pos_stats_keeper', 'Comp_stats_keeper', 'Age_stats_keeper', 'Born_stats_keeper', 'MP_stats_keeper', 'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt_stats_keeper', 'PKA', 'PKsv', 'PKm', 'Rk_stats_shooting', 'Nation_stats_shooting', 'Pos_stats_shooting', 'Comp_stats_shooting', 'Age_stats_shooting', 'Born_stats_shooting', '90s_stats_shooting', 'Gls_stats_shooting', 'Sh

# Data Loading V2
Enhanced pipeline with ELO ratings and player statistics

In [4]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

BASE = r'C:\Project\FIFA_World_Cup_2026'

# Load all datasets
results = pd.read_csv(os.path.join(BASE, 'Data', 'results.csv'))
goalscorers = pd.read_csv(os.path.join(BASE, 'Data', 'goalscorers.csv'))
elo = pd.read_csv(os.path.join(BASE, 'Data', 'elo_ratings_wc2026.csv'))
players = pd.read_csv(os.path.join(BASE, 'Data', 'players_data-2025_2026.csv'))
player_elo = pd.read_csv(os.path.join(BASE, 'Data', 'players.csv'))

# Convert dates
results['date'] = pd.to_datetime(results['date'])

# Filter to 2018+
results = results[results['date'] >= '2018-01-01']

# Drop missing scores
results = results.dropna(subset=['home_score', 'away_score'])

print("Results shape:", results.shape)
print("ELO shape:", elo.shape)
print("Players shape:", players.shape)
print("Player ELO shape:", player_elo.shape)

Results shape: (8068, 9)
ELO shape: (4683, 23)
Players shape: (2839, 102)
Player ELO shape: (70378, 16)


## Add Target Variable and Tournament Weights

In [5]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 1
    elif row['home_score'] < row['away_score']:
        return -1
    else:
        return 0

results['result'] = results.apply(get_result, axis=1)

tournament_weights = {
    'FIFA World Cup': 3.0,
    'FIFA World Cup qualification': 2.0,
    'UEFA Euro': 2.0,
    'UEFA Euro qualification': 1.5,
    'Copa América': 2.0,
    'Africa Cup of Nations': 2.0,
    'Friendly': 0.5,
    'Confederations Cup': 1.5
}
results['weight'] = results['tournament'].map(tournament_weights).fillna(1.0)

print("Target variable and weights added!")
print(results['result'].value_counts())

Target variable and weights added!
result
 1    3853
-1    2354
 0    1861
Name: count, dtype: int64


## Get Latest ELO Rating Per Team

In [6]:
# Get the most recent ELO rating for each team
latest_elo = elo.sort_values('year', ascending=False).groupby('country').first().reset_index()
latest_elo = latest_elo[['country', 'rating', 'wins', 'losses', 'draws', 
                          'goals_for', 'goals_against', 'confederation']]
latest_elo.columns = ['team', 'elo_rating', 'elo_wins', 'elo_losses', 
                      'elo_draws', 'elo_goals_for', 'elo_goals_against', 'confederation']

# Calculate win rate from ELO data
latest_elo['elo_win_rate'] = latest_elo['elo_wins'] / (
    latest_elo['elo_wins'] + latest_elo['elo_losses'] + latest_elo['elo_draws'] + 1
)

# Calculate goal difference
latest_elo['elo_goal_diff'] = latest_elo['elo_goals_for'] - latest_elo['elo_goals_against']

print("Latest ELO shape:", latest_elo.shape)
print(latest_elo[['team', 'elo_rating', 'elo_win_rate']].head(10))

Latest ELO shape: (48, 10)
                     team  elo_rating  elo_win_rate
0                 Algeria        1743      0.448476
1               Argentina        2113      0.549550
2               Australia        1783      0.512579
3                 Austria        1827      0.427252
4                 Belgium        1867      0.444698
5  Bosnia and Herzegovina        1594      0.371429
6                  Brazil        1984      0.628518
7                  Canada        1784      0.386503
8              Cape Verde        1549      0.373016
9                Colombia        1975      0.408320


## Build Player Strength Scores by Position

In [7]:
# Clean nation column - extract 3 letter code
players['nation_code'] = players['Nation'].str.split().str[-1].str.upper()

# Position mapping
def get_position_group(pos):
    if pd.isna(pos):
        return 'unknown'
    pos = str(pos).upper()
    if 'GK' in pos:
        return 'GK'
    elif 'DF' in pos:
        return 'DF'
    elif 'MF' in pos:
        return 'MF'
    elif 'FW' in pos:
        return 'FW'
    else:
        return 'MF'

players['pos_group'] = players['Pos'].apply(get_position_group)

# Convert numeric columns
num_cols = ['Gls', 'Ast', 'Int', 'TklW', 'Crs', 'MP', 'Min']
for col in num_cols:
    players[col] = pd.to_numeric(players[col], errors='coerce').fillna(0)

# Filter out players with very few minutes
players = players[players['Min'] > 200]

print("Positions distribution:")
print(players['pos_group'].value_counts())

Positions distribution:
pos_group
MF    1029
DF     816
FW     255
GK     158
Name: count, dtype: int64


## Aggregate Player Stats by Nation

In [8]:
# Attacking strength — forwards and wingers
attack_stats = players[players['pos_group'].isin(['FW'])].groupby('nation_code').agg(
    attack_goals=('Gls', 'mean'),
    attack_assists=('Ast', 'mean'),
    attack_crosses=('Crs', 'mean')
).reset_index()

# Midfield strength
mid_stats = players[players['pos_group'] == 'MF'].groupby('nation_code').agg(
    mid_assists=('Ast', 'mean'),
    mid_tackles=('TklW', 'mean'),
    mid_interceptions=('Int', 'mean')
).reset_index()

# Defensive strength
def_stats = players[players['pos_group'] == 'DF'].groupby('nation_code').agg(
    def_tackles=('TklW', 'mean'),
    def_interceptions=('Int', 'mean'),
    def_clearances=('Int', 'mean')
).reset_index()

print("Attack stats shape:", attack_stats.shape)
print("Mid stats shape:", mid_stats.shape)
print("Def stats shape:", def_stats.shape)

Attack stats shape: (58, 4)
Mid stats shape: (84, 4)
Def stats shape: (85, 4)


## Map Nation Codes to Full Team Names

In [9]:
# Map 3-letter codes to full country names matching results.csv
nation_map = {
    'ENG': 'England', 'FRA': 'France', 'ESP': 'Spain', 'GER': 'Germany',
    'ITA': 'Italy', 'POR': 'Portugal', 'NED': 'Netherlands', 'BEL': 'Belgium',
    'BRA': 'Brazil', 'ARG': 'Argentina', 'URU': 'Uruguay', 'COL': 'Colombia',
    'MEX': 'Mexico', 'USA': 'United States', 'CAN': 'Canada', 'CRC': 'Costa Rica',
    'MAR': 'Morocco', 'SEN': 'Senegal', 'NGA': 'Nigeria', 'EGY': 'Egypt',
    'JPN': 'Japan', 'KOR': 'South Korea', 'AUS': 'Australia', 'IRN': 'Iran',
    'KSA': 'Saudi Arabia', 'HRV': 'Croatia', 'CHE': 'Switzerland', 'AUT': 'Austria',
    'SRB': 'Serbia', 'DNK': 'Denmark', 'POL': 'Poland', 'HUN': 'Hungary',
    'SVK': 'Slovakia', 'SVN': 'Slovenia', 'TUR': 'Turkey', 'ECU': 'Ecuador',
    'PAR': 'Paraguay', 'HON': 'Honduras', 'PAN': 'Panama', 'QAT': 'Qatar',
    'CMR': 'Cameroon', 'MLI': 'Mali', 'CIV': 'Ivory Coast', 'ZAF': 'South Africa',
    'NZL': 'New Zealand', 'UZB': 'Uzbekistan', 'JOR': 'Jordan'
}

for df in [attack_stats, mid_stats, def_stats]:
    df['team'] = df['nation_code'].map(nation_map)

# Merge all player stats
player_strength = attack_stats.merge(mid_stats, on=['nation_code', 'team'], how='outer')
player_strength = player_strength.merge(def_stats, on=['nation_code', 'team'], how='outer')
player_strength = player_strength.dropna(subset=['team'])

print("Player strength shape:", player_strength.shape)
print(player_strength.head())

Player strength shape: (40, 11)
   nation_code  attack_goals  attack_assists  attack_crosses       team  \
3          ARG      7.666667        1.750000       14.916667  Argentina   
5          AUS           NaN             NaN             NaN  Australia   
6          AUT      3.500000        1.500000        4.000000    Austria   
7          BEL      1.000000        1.000000       19.000000    Belgium   
11         BRA      6.555556        1.666667        9.777778     Brazil   

    mid_assists  mid_tackles  mid_interceptions  def_tackles  \
3      2.482759    19.275862          10.068966    21.650000   
5      1.500000    13.000000          17.000000    25.000000   
6      2.428571    22.500000          17.142857    20.083333   
7      1.962963    17.222222          14.296296    15.214286   
11     2.052632    17.394737          11.473684    17.346154   

    def_interceptions  def_clearances  
3           22.500000       22.500000  
5           27.000000       27.000000  
6           

## Save All Processed Data

In [10]:
# Save cleaned results
results.to_csv(os.path.join(BASE, 'Cleaned_Data', 'results_cleaned_v2.csv'), index=False)

# Save ELO data
latest_elo.to_csv(os.path.join(BASE, 'Cleaned_Data', 'elo_cleaned.csv'), index=False)

# Save player strength
player_strength.to_csv(os.path.join(BASE, 'Cleaned_Data', 'player_strength.csv'), index=False)

print("All V2 data saved!")
print("\nResults shape:", results.shape)
print("ELO shape:", latest_elo.shape)
print("Player strength shape:", player_strength.shape)

All V2 data saved!

Results shape: (8068, 11)
ELO shape: (48, 10)
Player strength shape: (40, 11)


## V3 Enhancements

In [11]:
# Remove friendlies from training data
competitive = results[~results['tournament'].str.contains('Friendly', na=False)].copy()

print("All matches:", len(results))
print("Competitive only:", len(competitive))
print("\nTournament types kept:")
print(competitive['tournament'].value_counts().head(15))

All matches: 8068
Competitive only: 5831

Tournament types kept:
tournament
FIFA World Cup qualification             1767
UEFA Nations League                       658
African Cup of Nations qualification      560
UEFA Euro qualification                   501
CONCACAF Nations League                   422
African Cup of Nations                    208
FIFA World Cup                            128
Gold Cup                                  124
AFC Asian Cup qualification               118
COSAFA Cup                                112
AFC Asian Cup                             102
UEFA Euro                                 102
Copa América                               86
AFF Championship                           78
CONCACAF Nations League qualification      68
Name: count, dtype: int64


## Recency Weights

In [12]:
# Give more weight to recent matches
def get_recency_weight(date):
    year = date.year
    if year >= 2025:
        return 3.0
    elif year >= 2023:
        return 2.0
    elif year >= 2021:
        return 1.5
    else:
        return 1.0

competitive['recency_weight'] = competitive['date'].apply(get_recency_weight)
competitive['final_weight'] = competitive['weight'] * competitive['recency_weight']

print("Recency weights applied!")
print(competitive['recency_weight'].value_counts())

Recency weights applied!
recency_weight
2.0    1790
1.0    1641
1.5    1521
3.0     879
Name: count, dtype: int64


## Head to Head Feature

In [13]:
def get_h2h(df, home, away, date, n=10):
    h2h = df[
        (((df['home_team'] == home) & (df['away_team'] == away)) |
         ((df['home_team'] == away) & (df['away_team'] == home))) &
        (df['date'] < date)
    ].tail(n)
    
    if len(h2h) == 0:
        return 0.5
    
    home_wins = 0
    for _, row in h2h.iterrows():
        if row['home_team'] == home and row['result'] == 1:
            home_wins += 1
        elif row['away_team'] == home and row['result'] == -1:
            home_wins += 1
    
    return home_wins / len(h2h)

print("H2H function defined!")

H2H function defined!


## Star Player ELO per Team

In [ ]:
# Get top 3 players ELO per nationality and average them
player_elo_clean = player_elo.copy()
player_elo_clean['elo'] = pd.to_numeric(player_elo_clean['elo'], errors='coerce')

# Map nationality codes to team names
nationality_map = {
    'England': 'England', 'France': 'France', 'Spain': 'Spain',
    'Germany': 'Germany', 'Italy': 'Italy', 'Portugal': 'Portugal',
    'Netherlands': 'Netherlands', 'Belgium': 'Belgium',
    'Brazil': 'Brazil', 'Argentina': 'Argentina', 'Uruguay': 'Uruguay',
    'Colombia': 'Colombia', 'Mexico': 'Mexico', 'United States': 'United States',
    'Canada': 'Canada', 'Japan': 'Japan', 'South Korea': 'South Korea',
    'Australia': 'Australia', 'Iran': 'Iran', 'Saudi Arabia': 'Saudi Arabia',
    'Morocco': 'Morocco', 'Senegal': 'Senegal', 'Egypt': 'Egypt',
    'Ghana': 'Ghana', 'Croatia': 'Croatia', 'Switzerland': 'Switzerland',
    'Austria': 'Austria', 'Norway': 'Norway', 'Scotland': 'Scotland',
    'Sweden': 'Sweden', 'Turkey': 'Turkey', 'Ecuador': 'Ecuador',
    'Paraguay': 'Paraguay', 'Algeria': 'Algeria', 'Tunisia': 'Tunisia',
    'Ivory Coast': 'Ivory Coast', 'South Africa': 'South Africa',
    'Panama': 'Panama', 'Qatar': 'Qatar', 'Uzbekistan': 'Uzbekistan',
    'Jordan': 'Jordan', 'Iraq': 'Iraq', 'New Zealand': 'New Zealand',
    'Bosnia and Herzegovina': 'Bosnia and Herzegovina'
}

player_elo_clean['team'] = player_elo_clean['nationality'].map(nationality_map)

# Get top 3 players per team by ELO
star_elo = player_elo_clean.dropna(subset=['team', 'elo'])
star_elo = star_elo.sort_values('elo', ascending=False)
star_elo = star_elo.groupby('team').head(3)
star_elo = star_elo.groupby('team')['elo'].mean().reset_index()
star_elo.columns = ['team', 'star_player_elo']

print("Star player ELO shape:", star_elo.shape)
print(star_elo.sort_values('star_player_elo', ascending=False).head(10))

## Squad Age Feature

In [ ]:
# Average squad age per nation
players['nation_code'] = players['Nation'].str.split().str[-1].str.upper()

nation_map = {
    'ENG': 'England', 'FRA': 'France', 'ESP': 'Spain', 'GER': 'Germany',
    'ITA': 'Italy', 'POR': 'Portugal', 'NED': 'Netherlands', 'BEL': 'Belgium',
    'BRA': 'Brazil', 'ARG': 'Argentina', 'URU': 'Uruguay', 'COL': 'Colombia',
    'MEX': 'Mexico', 'USA': 'United States', 'CAN': 'Canada', 'CRC': 'Costa Rica',
    'MAR': 'Morocco', 'SEN': 'Senegal', 'NGA': 'Nigeria', 'EGY': 'Egypt',
    'JPN': 'Japan', 'KOR': 'South Korea', 'AUS': 'Australia', 'IRN': 'Iran',
    'KSA': 'Saudi Arabia', 'HRV': 'Croatia', 'CHE': 'Switzerland', 'AUT': 'Austria',
    'SRB': 'Serbia', 'DNK': 'Denmark', 'POL': 'Poland', 'HUN': 'Hungary',
    'SVK': 'Slovakia', 'SVN': 'Slovenia', 'TUR': 'Turkey', 'ECU': 'Ecuador',
    'PAR': 'Paraguay', 'QAT': 'Qatar', 'CMR': 'Cameroon', 'MLI': 'Mali',
    'CIV': 'Ivory Coast', 'ZAF': 'South Africa', 'NZL': 'New Zealand',
    'UZB': 'Uzbekistan', 'JOR': 'Jordan', 'NOR': 'Norway', 'SCO': 'Scotland',
    'SWE': 'Sweden', 'ALG': 'Algeria', 'TUN': 'Tunisia', 'GHA': 'Ghana',
    'BIH': 'Bosnia and Herzegovina', 'CZE': 'Czech Republic', 'PAN': 'Panama',
    'IRQ': 'Iraq', 'HAI': 'Haiti'
}

players['team'] = players['nation_code'].map(nation_map)
players['Age'] = pd.to_numeric(players['Age'], errors='coerce')

squad_age = players.dropna(subset=['team', 'Age'])
squad_age = squad_age.groupby('team')['Age'].mean().reset_index()
squad_age.columns = ['team', 'avg_squad_age']

# Peak age is 26-28 — create peak age score
squad_age['age_score'] = 1 - abs(squad_age['avg_squad_age'] - 27) / 10

print("Squad age shape:", squad_age.shape)
print(squad_age.sort_values('age_score', ascending=False).head(10))

## Confederation Strength

In [18]:
elo_conf = elo[['country', 'confederation']].drop_duplicates()
elo_conf.columns = ['team', 'confederation']
elo_conf['conf_strength'] = elo_conf['confederation'].map(confederation_strength).fillna(0.7)

print("Confederation strength:")
print(elo_conf.groupby('confederation')['conf_strength'].first())

Confederation strength:
confederation
AFC         0.75
CAF         0.75
CONCACAF    0.65
CONMEBOL    0.90
OFC         0.50
UEFA        1.00
Name: conf_strength, dtype: float64


## Save All V3 Data

In [19]:
competitive.to_csv(os.path.join(BASE, 'Cleaned_Data', 'results_competitive_v3.csv'), index=False)
star_elo.to_csv(os.path.join(BASE, 'Cleaned_Data', 'star_player_elo.csv'), index=False)
squad_age.to_csv(os.path.join(BASE, 'Cleaned_Data', 'squad_age.csv'), index=False)
elo_conf.to_csv(os.path.join(BASE, 'Cleaned_Data', 'confederation_strength.csv'), index=False)

print("All V3 data saved!")
print("Competitive matches:", len(competitive))
print("Star player ELO teams:", len(star_elo))
print("Squad age teams:", len(squad_age))

All V3 data saved!
Competitive matches: 5831
Star player ELO teams: 42
Squad age teams: 48
